# Hate / Non-Hate Vision Dataset Preprocessing

This notebook builds a binary video manifest from `dataset.csv` and `final_video_labels.csv`, then extracts perceptually unique frames from each available video.

Outputs:
- `E:/m-hvc/datasets/old-dataset/video_df.csv`
- `E:/m-hvc/datasets/old-dataset/video_df_label_conflicts.csv`
- `E:/m-hvc/datasets/old-dataset/video_df_missing_videos.csv`
- `E:/m-hvc/datasets/old-dataset/frames_unique/{source_video_id}/*.jpg`


In [2]:
from pathlib import Path
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
import os
import shutil
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


C:\Users\Rasheek\.conda\envs\fyp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PROJECT_ROOT = Path(r"E:/m-hvc")
DATASET_DIR = PROJECT_ROOT / "datasets" / "old-dataset"
VIDEO_ROOT = DATASET_DIR / "video"
FRAMES_ROOT = DATASET_DIR / "frames_unique"

DATASET_CSV = DATASET_DIR / "dataset.csv"
FINAL_LABELS_CSV = DATASET_DIR / "final_video_labels.csv"
VIDEO_DF_CSV = DATASET_DIR / "video_df.csv"
CONFLICT_REPORT_CSV = DATASET_DIR / "video_df_label_conflicts.csv"
MISSING_REPORT_CSV = DATASET_DIR / "video_df_missing_videos.csv"

CFG = {
    "seed": 42,
    "max_unique_frames_per_video": 64,
    "image_size": 256,
    "jpeg_quality": 95,
    "uniqueness_threshold": 4.0,
    "hash_size": 8,
    "hash_hamming_threshold": 4,
    "overwrite_existing_frames": False,
    "num_workers": max(1, min(8, (os.cpu_count() or 4) - 1)),
    "in_flight_factor": 2,
    "save_every": 100,
}

VALID_VIDEO_SUFFIXES = {".mp4", ".webm", ".avi", ".mov", ".mkv"}

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)

print(f"dataset dir: {DATASET_DIR}")
print(f"video root exists: {VIDEO_ROOT.exists()} ({VIDEO_ROOT})")
print(f"frame output root: {FRAMES_ROOT}")
print(f"workers: {CFG['num_workers']}")


dataset dir: E:\m-hvc\datasets\old-dataset
video root exists: True (E:\m-hvc\datasets\old-dataset\video)
frame output root: E:\m-hvc\datasets\old-dataset\frames_unique
workers: 8


## Build `video_df.csv`

`final_video_labels.csv` is treated as authoritative for overlapping videos. `dataset.csv` only contributes additional resolvable videos that are not already present in `final_video_labels.csv`.

In [4]:
def normalize_source_video_id(value):
    text = str(value).strip()
    for suffix in VALID_VIDEO_SUFFIXES:
        if text.lower().endswith(suffix):
            return text[: -len(suffix)]
    return text


def normalize_video_file_name(value):
    source_video_id = normalize_source_video_id(value)
    return f"{source_video_id}.mp4"


def build_video_lookup(video_root):
    extension_priority = {".mp4": 0, ".webm": 1, ".avi": 2, ".mov": 3, ".mkv": 4}
    best_files = {}

    for path in video_root.iterdir():
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix not in VALID_VIDEO_SUFFIXES:
            continue

        rank = extension_priority.get(suffix, 99)
        current = best_files.get(path.stem)
        if current is None or rank < current[0]:
            best_files[path.stem] = (rank, path)

    return {source_video_id: path for source_video_id, (_, path) in best_files.items()}


video_lookup = build_video_lookup(VIDEO_ROOT)
print(f"available videos: {len(video_lookup):,}")


available videos: 3,429


In [5]:
raw_dataset_df = pd.read_csv(DATASET_CSV)
raw_final_labels_df = pd.read_csv(FINAL_LABELS_CSV)

dataset_df = raw_dataset_df.copy()
dataset_df["source_video_id"] = dataset_df["video_file_name"].map(normalize_source_video_id)
dataset_df["video_file_name"] = dataset_df["source_video_id"].map(normalize_video_file_name)
dataset_df["label"] = dataset_df["Action"].astype(int)
dataset_df["label_source"] = "dataset.csv"
dataset_df["source"] = "old_dataset"
dataset_df["transcription"] = dataset_df.get("transcription", "")

dataset_conflicts_df = (
    dataset_df.groupby("source_video_id")
    .agg(
        labels=("label", lambda values: sorted(set(map(int, values)))),
        row_count=("label", "size"),
        video_file_name=("video_file_name", "first"),
    )
    .reset_index()
)
dataset_conflicts_df = dataset_conflicts_df[dataset_conflicts_df["labels"].map(len) > 1].copy()
dataset_conflicts_df["conflict_type"] = "dataset_duplicate_label_conflict"

# If dataset.csv has duplicate rows, collapse deterministically. For internal conflicts,
# prefer hate=1 because false negatives are more damaging for this task.
dataset_collapsed_df = (
    dataset_df.sort_values(["source_video_id", "label"], ascending=[True, False])
    .groupby("source_video_id", as_index=False)
    .agg(
        video_file_name=("video_file_name", "first"),
        label=("label", "max"),
        label_source=("label_source", "first"),
        source=("source", "first"),
        transcription=("transcription", "first"),
    )
)

final_labels_df = raw_final_labels_df.copy()
final_labels_df["source_video_id"] = final_labels_df["video_id"].map(normalize_source_video_id)
final_labels_df["video_file_name"] = final_labels_df["source_video_id"].map(normalize_video_file_name)
final_labels_df["label"] = final_labels_df["label"].astype(int)
final_labels_df["label_source"] = "final_video_labels.csv"
final_labels_df["source"] = final_labels_df.get("source", "final_video_labels")
final_labels_df["transcription"] = ""
final_labels_df = final_labels_df[
    ["source_video_id", "video_file_name", "label", "label_source", "source", "transcription"]
].drop_duplicates("source_video_id")

overlap_labels_df = dataset_collapsed_df[["source_video_id", "label"]].merge(
    final_labels_df[["source_video_id", "label"]],
    on="source_video_id",
    suffixes=("_dataset", "_final"),
)
overlap_conflicts_df = overlap_labels_df[
    overlap_labels_df["label_dataset"] != overlap_labels_df["label_final"]
].copy()
overlap_conflicts_df["conflict_type"] = "dataset_vs_final_label_conflict"

dataset_only_df = dataset_collapsed_df[
    ~dataset_collapsed_df["source_video_id"].isin(final_labels_df["source_video_id"])
].copy()

combined_df = pd.concat([final_labels_df, dataset_only_df], ignore_index=True)
combined_df["video_path"] = combined_df["source_video_id"].map(video_lookup)
combined_df["has_video_file"] = combined_df["video_path"].notna()

missing_videos_df = combined_df[~combined_df["has_video_file"]].copy()
video_df = combined_df[combined_df["has_video_file"]].copy()
video_df["video_path"] = video_df["video_path"].map(lambda path: str(Path(path)))
video_df["label"] = video_df["label"].astype(int)
video_df = video_df.sort_values(["label_source", "source_video_id"]).reset_index(drop=True)

assert set(video_df["label"].unique()).issubset({0, 1})
assert video_df["source_video_id"].is_unique
assert video_df["video_path"].map(lambda value: Path(value).exists()).all()

conflict_reports = []
if not dataset_conflicts_df.empty:
    conflict_reports.append(dataset_conflicts_df)
if not overlap_conflicts_df.empty:
    conflict_reports.append(overlap_conflicts_df)

if conflict_reports:
    pd.concat(conflict_reports, ignore_index=True, sort=False).to_csv(CONFLICT_REPORT_CSV, index=False)
else:
    pd.DataFrame(columns=["source_video_id", "conflict_type"]).to_csv(CONFLICT_REPORT_CSV, index=False)

missing_videos_df.to_csv(MISSING_REPORT_CSV, index=False)
video_df.to_csv(VIDEO_DF_CSV, index=False)

print(f"final manifest rows: {len(video_df):,}")
print(f"missing videos dropped: {len(missing_videos_df):,}")
print(f"dataset duplicate label conflicts: {len(dataset_conflicts_df):,}")
print(f"dataset/final overlap conflicts: {len(overlap_conflicts_df):,}")
display(video_df.head())
display(video_df["label"].value_counts().rename_axis("label").reset_index(name="count"))


final manifest rows: 3,283
missing videos dropped: 107
dataset duplicate label conflicts: 8
dataset/final overlap conflicts: 3


,source_video_id,video_file_name,label,label_source,source,transcription,video_path,has_video_file
0,R_hate_video_099,R_hate_video_099.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
1,R_hate_video_100,R_hate_video_100.mp4,1,dataset.csv,old_dataset,Parinkar. Hai. Haiya.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
2,R_hate_video_101,R_hate_video_101.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
3,R_hate_video_102,R_hate_video_102.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
4,R_hate_video_103,R_hate_video_103.mp4,1,dataset.csv,old_dataset,The darkest.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True


,label,count
0,0,1948
1,1,1335


## Extract Unique Frames

This keeps frames that differ from the last saved frame by both average pixel difference and perceptual hash distance. The cap prevents runaway storage use while still preserving more temporal coverage than a fixed small uniform sample.

In [6]:
try:
    import cv2
except ImportError as exc:
    raise ImportError(
        "OpenCV is required for frame extraction. Install it with `pip install opencv-python` "
        "or the environment-specific package manager before running this section."
    ) from exc


def clear_directory(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def average_hash(gray_frame, hash_size):
    resized = cv2.resize(gray_frame, (hash_size, hash_size), interpolation=cv2.INTER_AREA)
    return resized >= resized.mean()


def hamming_distance(hash_a, hash_b):
    return int(np.count_nonzero(hash_a != hash_b))


def is_unique_frame(gray_frame, previous_gray, current_hash, previous_hash, cfg):
    if previous_gray is None or previous_hash is None:
        return True

    resized_previous = cv2.resize(previous_gray, (gray_frame.shape[1], gray_frame.shape[0]), interpolation=cv2.INTER_AREA)
    mean_abs_diff = float(np.mean(cv2.absdiff(gray_frame, resized_previous)))
    hash_diff = hamming_distance(current_hash, previous_hash)
    return mean_abs_diff >= cfg["uniqueness_threshold"] and hash_diff >= cfg["hash_hamming_threshold"]


def write_frame(frame, output_dir, frame_index, saved_index, image_size, jpeg_quality):
    resized = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_AREA)
    frame_path = output_dir / f"{saved_index:04d}_src{frame_index:06d}.jpg"
    write_ok = cv2.imwrite(
        str(frame_path),
        resized,
        [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)],
    )
    return frame_path if write_ok else None


def extract_unique_frames_task(task):
    source_video_id = str(task["source_video_id"])
    video_path = Path(task["video_path"])
    output_dir = Path(task["frame_dir"])
    max_frames = int(task["max_unique_frames_per_video"])
    image_size = int(task["image_size"])
    jpeg_quality = int(task["jpeg_quality"])
    overwrite = bool(task["overwrite_existing_frames"])
    cfg = task["cfg"]

    if overwrite:
        clear_directory(output_dir)
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(output_dir.glob("*.jpg"))
        if existing:
            return {
                "source_video_id": source_video_id,
                "video_path": str(video_path),
                "frame_dir": str(output_dir),
                "ok": True,
                "frame_status": "cached",
                "num_frames_saved": len(existing),
                "used_cache": True,
            }

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return {
            "source_video_id": source_video_id,
            "video_path": str(video_path),
            "frame_dir": str(output_dir),
            "ok": False,
            "frame_status": "open_failed",
            "num_frames_saved": 0,
            "used_cache": False,
        }

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    saved_paths = []
    previous_gray = None
    previous_hash = None
    frame_index = 0

    while len(saved_paths) < max_frames:
        ok, frame = capture.read()
        if not ok or frame is None:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        current_hash = average_hash(gray, int(cfg["hash_size"]))

        if is_unique_frame(gray, previous_gray, current_hash, previous_hash, cfg):
            frame_path = write_frame(
                frame=frame,
                output_dir=output_dir,
                frame_index=frame_index,
                saved_index=len(saved_paths),
                image_size=image_size,
                jpeg_quality=jpeg_quality,
            )
            if frame_path is not None:
                saved_paths.append(frame_path)
                previous_gray = gray
                previous_hash = current_hash

        frame_index += 1

    capture.release()

    if not saved_paths:
        return {
            "source_video_id": source_video_id,
            "video_path": str(video_path),
            "frame_dir": str(output_dir),
            "ok": False,
            "frame_status": "no_frames_saved",
            "num_frames_saved": 0,
            "used_cache": False,
            "frame_count": frame_count,
            "fps": fps,
            "width": width,
            "height": height,
            "duration_sec": (frame_count / fps) if fps and fps > 0 else np.nan,
        }

    return {
        "source_video_id": source_video_id,
        "video_path": str(video_path),
        "frame_dir": str(output_dir),
        "ok": True,
        "frame_status": "ok",
        "num_frames_saved": len(saved_paths),
        "used_cache": False,
        "frame_count": frame_count,
        "fps": fps,
        "width": width,
        "height": height,
        "duration_sec": (frame_count / fps) if fps and fps > 0 else np.nan,
    }


In [7]:
def build_frame_task(row):
    return {
        "source_video_id": str(row.source_video_id),
        "video_path": str(row.video_path),
        "frame_dir": str(FRAMES_ROOT / str(row.source_video_id)),
        "max_unique_frames_per_video": CFG["max_unique_frames_per_video"],
        "image_size": CFG["image_size"],
        "jpeg_quality": CFG["jpeg_quality"],
        "overwrite_existing_frames": CFG["overwrite_existing_frames"],
        "cfg": CFG,
    }


def persist_frame_results(results):
    if not results:
        return video_df.copy()

    results_df = pd.DataFrame(results).drop_duplicates("source_video_id", keep="last")
    merged_df = video_df.drop(
        columns=[
            "frame_dir",
            "ok",
            "frame_status",
            "num_frames_saved",
            "used_cache",
            "frame_count",
            "fps",
            "width",
            "height",
            "duration_sec",
        ],
        errors="ignore",
    ).merge(results_df, on=["source_video_id", "video_path"], how="left")
    merged_df.to_csv(VIDEO_DF_CSV, index=False)
    return merged_df


def run_frame_extraction(input_df, limit=None):
    candidate_df = input_df.copy()
    if limit is not None:
        candidate_df = candidate_df.head(int(limit)).copy()

    tasks = [build_frame_task(row) for row in candidate_df.itertuples(index=False)]
    results = []
    max_in_flight = max(CFG["num_workers"], CFG["num_workers"] * CFG["in_flight_factor"])
    task_iter = iter(tasks)

    progress_bar = tqdm(total=len(tasks), desc="extract_unique_frames")
    with ThreadPoolExecutor(max_workers=CFG["num_workers"]) as executor:
        future_to_task = {}

        while len(future_to_task) < min(max_in_flight, len(tasks)):
            task = next(task_iter, None)
            if task is None:
                break
            future_to_task[executor.submit(extract_unique_frames_task, task)] = task

        while future_to_task:
            done, _ = wait(future_to_task, return_when=FIRST_COMPLETED)
            for future in done:
                task = future_to_task.pop(future)
                try:
                    result = future.result()
                except Exception as exc:
                    result = {
                        "source_video_id": task["source_video_id"],
                        "video_path": task["video_path"],
                        "frame_dir": task["frame_dir"],
                        "ok": False,
                        "frame_status": f"exception: {type(exc).__name__}: {exc}",
                        "num_frames_saved": 0,
                        "used_cache": False,
                    }

                results.append(result)
                progress_bar.update(1)

                if len(results) % CFG["save_every"] == 0:
                    persist_frame_results(results)

                next_task = next(task_iter, None)
                if next_task is not None:
                    future_to_task[executor.submit(extract_unique_frames_task, next_task)] = next_task

    progress_bar.close()
    return persist_frame_results(results)


Run the smoke test first. If it succeeds, run the full extraction cell.

In [7]:
smoke_df = run_frame_extraction(video_df, limit=10)
display(smoke_df[["source_video_id", "label", "frame_status", "num_frames_saved", "frame_dir"]].head(10))


extract_unique_frames: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 88.13it/s]


,source_video_id,label,frame_status,num_frames_saved,frame_dir
0,R_hate_video_099,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
1,R_hate_video_100,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
2,R_hate_video_101,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
3,R_hate_video_102,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
4,R_hate_video_103,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
5,R_hate_video_104,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
6,R_hate_video_105,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
7,R_hate_video_106,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
8,R_hate_video_107,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...
9,R_hate_video_108,1,cached,64.0,E:\m-hvc\datasets\old-dataset\frames_unique\R_...


In [ ]:
# Run this cell after the smoke test passes.
video_df = pd.read_csv(VIDEO_DF_CSV)
video_df = run_frame_extraction(video_df)

display(video_df["frame_status"].value_counts(dropna=False).rename_axis("frame_status").reset_index(name="count"))
display(video_df.groupby("label")["num_frames_saved"].agg(["count", "min", "median", "max"]))


extract_unique_frames: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 3281/3283 [00:19<00:00, 1695.86it/s]

In [8]:
final_video_df = pd.read_csv(VIDEO_DF_CSV)

assert set(final_video_df["label"].dropna().astype(int).unique()).issubset({0, 1})
assert final_video_df["source_video_id"].is_unique
assert final_video_df["video_path"].map(lambda value: Path(value).exists()).all()

if "ok" in final_video_df.columns:
    successful = final_video_df[final_video_df["ok"] == True].copy()
else:
    successful = final_video_df.iloc[0:0].copy()
if not successful.empty:
    assert successful["frame_dir"].map(lambda value: Path(value).exists()).all()
    assert (successful["num_frames_saved"].fillna(0).astype(int) > 0).all()

print(f"manifest saved: {VIDEO_DF_CSV}")
print(f"rows: {len(final_video_df):,}")
print(f"successful frame dirs: {len(successful):,}")
display(final_video_df.head())


manifest saved: E:\m-hvc\datasets\old-dataset\video_df.csv
rows: 3,283
successful frame dirs: 0


,source_video_id,video_file_name,label,label_source,source,transcription,video_path,has_video_file
0,R_hate_video_099,R_hate_video_099.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
1,R_hate_video_100,R_hate_video_100.mp4,1,dataset.csv,old_dataset,Parinkar. Hai. Haiya.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
2,R_hate_video_101,R_hate_video_101.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
3,R_hate_video_102,R_hate_video_102.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
4,R_hate_video_103,R_hate_video_103.mp4,1,dataset.csv,old_dataset,The darkest.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True
